[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equiphysics/education/blob/main/labs/Horse-Artery-Lab/Horse_Artery_Lab.ipynb)

# Horse Artery Lab
### Blood pressure from heart to hoof, in a computer horse

This notebook runs a computer model of a horse's arteries: 117 arteries from the heart to the head, the four legs, and the gut, with a beating heart at one end and 62 tissue beds at the other. The anatomy and settings come from a study by veterinarians and engineers at Ghent University and EPFL (Vera et al., 2019, *PLoS ONE*). EquiPhysics rewrote their model in Python so you can run it here.

**How to use it.** Run the cells in order: click a cell and press Shift+Enter, or use *Runtime → Run all*. The first cell builds the model. Each new horse takes about half a minute to a minute to run. A horse you have already run comes back instantly. If Colab warns that the notebook was not written by Google, click *Run anyway*. To keep your changes, use *File → Save a copy in Drive*.

In [ ]:
#@title Setup: run this cell first (it builds the horse model) { display-mode: "form" }
# The model code (package horse1d) is written to files here, then imported.
import os
os.makedirs('horse1d', exist_ok=True)
FILES = {}
FILES['horse1d/__init__.py'] = r''''''
FILES['horse1d/network.py'] = r'''"""The equine arterial tree from Vera et al. (2019), PLoS ONE 14(8): e0221425.

Table 1 of the paper lists 117 segments: lengths, branching angles, tapered lumen
diameters, and distensibilities. The branching (which segment feeds which) is
read from Fig 1 and from equine anatomy. The paper's data are CC BY 4.0.

Angle convention (from Fig 1, horse facing left): 0 deg points toward the tail,
90 deg straight up, 180 deg toward the head, 270 deg straight down. A segment of
length L at angle phi rises L*sin(phi).
"""

# seg | name | angle (deg) | mean length (mm) | SD | mean diameter (mm) | SD |
#     proximal lumen diameter (mm) | distal lumen diameter (mm) | distensibility (1e-3 / mmHg)
# "-" means no value in the paper.
_TABLE1 = """
1|Aorta ascendens 1|90|18|9|68|6|68.18|67.39|6.85
2|A coronaria sinistra|180|20|-|19|6|23.638|12.10|3.17
3|A coronaria dextra|0|20|-|19|6|23.638|12.10|3.17
4|Aorta ascendens 2|90|64|9|67|6|67.22|66.82|6.80
5|Truncus brachiocephalicus 1|160|38|26|39|9|38.882|38.88|4.90
6|A subclavia sinistra 1|155|39|24|26|11|25.332|25.33|3.79
7|Truncus costocervicalis sinister|90|10|-|11|5|12.102|10.50|2.34
8|A subclavia sinistra 2|155|19|13|24|8|24.11|24.11|3.68
9|A cervicalis profunda sinistra|115|70|-|3|1|2.82|2.82|0.87
10|A subclavia sinistra 3|155|14|8|24|8|24.11|24.11|3.68
11|A vertebralis sinistra|135|95|-|12|5|14.17|9.98|2.45
12|A subclavia sinistra 4|240|40|6|23|6|22.89|22.89|3.57
13|A thoracica interna sinistra|280|137|-|12|4|12.11|12.11|2.44
14|A subclavia sinistra 5|240|7|8|23|6|22.89|22.89|3.57
15|A cervicalis superficialis sinistra|180|80|-|8|2|10.31|4.79|1.90
16|A axillaris sinistra 1|310|62|25|21|6|21.50|20.37|3.38
17|A suprascapularis sinistra|70|15|-|6|0|7.68|3.52|1.59
18|A axillaris sinistra 2|310|58|45|20|8|19.90|19.90|3.28
19|A subscapularis sinistra|10|10|-|17|7|16.80|16.80|2.96
20|A axillaris sinistra 3|310|23|18|18|5|18.02|18.02|3.09
21|A circumflexa humeri cranialis sinistra|230|30|-|5|2|6.11|2.79|1.39
22|A axillaris sinistra 4|270|70|28|17|3|17.05|17.05|2.99
23|A profunda brachii sinistra|325|5|-|11|5|11.281|9.60|2.23
24|A axillaris sinistra 5|270|82|11|14|1|15.35|12.79|2.67
25|A collateralis ulnaris sinistra|320|5|-|6|2|8.60|4.92|1.75
26|A mediana sinistra|270|145|-|7|3|10.01|5.30|1.90
27|Truncus brachiocephalicus 2|160|50|-|39|-|38.88|38.88|4.90
28|Truncus costocervicalis dexter|90|10|-|11|6|14.38|7.07|2.34
29|Truncus brachiocephalicus 3|160|10|-|39|-|38.88|38.88|4.90
30|A cervicalis profunda dextra|115|70|-|3|2|2.81|2.81|0.87
31|Truncus brachiocephalicus 4|160|10|-|39|-|38.88|38.88|4.90
32|A vertebralis dextra|135|95|-|12|5|16.95|7.12|2.54
33|Truncus brachiocephalicus 5|160|20|-|26|-|26.96|25.00|4.24
34|A subclavia dextra 1|240|30|-|26|11|25.00|25.00|3.76
35|A thoracica interna dextra|280|137|-|11|5|12.12|12.10|2.44
36|A cervicalis superficialis dextra|180|80|-|3|1|8.07|8.02|1.90
37|A axillaris dextra 1|310|62|25|21|6|20.95|20.94|3.38
38|A suprascapularis dextra|70|15|-|6|0|6.43|5.47|1.59
39|A axillaris dextra 2|310|58|45|20|8|19.90|19.90|3.28
40|A subscapularis dextra|10|10|-|17|7|16.80|16.80|2.96
41|A axillaris dextra 3|310|23|18|18|5|18.02|18.02|3.09
42|A circumflexa humeri cranialis dextra|230|30|-|5|2|6.115|2.79|1.39
43|A axillaris dextra 4|270|70|28|17|3|17.05|17.05|2.99
44|A profunda brachii dextra|325|5|-|11|5|10.47|10.47|2.23
45|A axillaris dextra 5|270|82|11|14|1|17.05|10.42|2.67
46|A collateralis ulnaris dextra|320|5|-|6|2|8.60|4.92|1.75
47|A mediana dextra|270|145|-|7|3|10.00|5.30|1.90
48|Truncus bicaroticus|165|78|91|22|19|22.06|21.26|3.45
49|A carotis communis sinistra|110|710|59|12|1|13.45|10.35|2.42
50|A carotis interna sinistra|90|120|-|4|1|4.43|3.53|1.25
51|A occipitalis sinistra|60|45|-|5|2|4.94|4.13|1.35
52|A carotis externa sinistra 1|130|65|41|10|3|10.04|10.04|2.18
53|Truncus linguofacialis sinister|180|80|-|6|3|6.54|5.50|1.60
54|A carotis externa sinistra 2|90|41|11|9|2|9.22|8.83|2.04
55|Ramus massetericus sinister|255|20|-|2|1|2.51|1.21|0.82
56|A carotis externa sinistra 3|90|13|3|9|2|8.56|8.55|1.98
57|A auricularis caudalis sinistra|55|10|-|3|2|3.953|1.74|1.07
58|A carotis externa sinistra 4|90|23|3|7|3|8.56|5.15|1.76
59|A temporalis superficialis sinistra|60|20|-|2|2|3.44|2.48|1.05
58b|A carotis externa sinistra 5|90|50|-|5|-|5.15|5.15|1.46
60|A carotis communis dextra|110|710|59|12|1|13.45|10.35|2.42
61|A carotis interna dextra|90|120|-|4|1|4.43|3.53|1.25
62|A occipitalis dextra|60|45|-|5|2|4.94|4.13|1.35
63|A carotis externa dextra 1|130|65|41|10|3|10.04|10.04|2.18
64|Truncus linguofacialis dexter|180|80|-|6|3|6.54|5.50|1.60
65|A carotis externa dextra 2|90|41|11|9|2|9.22|8.83|2.04
66|Ramus massetericus dexter|255|20|-|2|1|2.51|1.21|0.82
67|A carotis externa dextra 3|90|13|3|9|2|8.56|8.55|1.98
68|A auricularis caudalis dextra|55|10|-|3|2|3.95|1.74|1.07
69|A carotis externa dextra 4|90|23|3|7|3|8.56|5.15|1.76
70|A temporalis superficialis dextra|60|20|-|2|2|3.44|2.48|1.05
69b|A carotis externa dextra 5|90|50|-|5|-|5.15|5.15|1.46
71|Arcus aortae (lig. art. Botalli)|45|77|26|54|10|58.07|49.53|5.97
72|Aorta descendens 1|45|105|5|45|11|48.21|40.62|5.32
73|A broncho-oesophagea|270|66|-|11|2|13.71|6.31|2.26
74|Aorta descendens 2|0|486|69|36|1|38.44|32.28|4.64
75|A coeliaca|270|20|-|15|3|19.31|9.24|2.78
76|Aorta descendens 3|0|48|26|29|6|28.70|28.70|4.09
77|A mesenterica cranialis|270|20|-|11|7|13.48|7.56|2.29
78|Aorta descendens 4|0|30|35|29|10|28.70|28.70|4.09
79|A renalis dextra|270|30|-|11|2|13.26|7.21|2.26
80|Aorta descendens 5|0|20|19|29|5|28.70|28.70|4.09
81|A renalis sinistra|270|30|-|11|2|13.26|7.21|2.26
82|Aorta descendens 6|0|104|60|29|6|28.70|28.70|4.09
83|A ovarica sinistra|270|300|-|3|1|2.76|2.76|1.00
84|A ovarica dextra|270|300|-|3|1|2.76|2.76|1.00
85|Aorta descendens 7|0|40|27|29|6|28.70|28.70|4.09
86|A mesenterica caudalis|270|10|-|8|3|9.78|4.63|1.85
87|Aorta descendens 8|0|63|12|29|6|28.70|28.70|4.09
88|A iliaca interna sinistra|330|30|10|18|6|20.19|15.12|0.92
89|A iliaca externa sinistra 1|315|30|-|17|2|16.93|16.24|0.88
90|A circumflexa ilium profunda sinistra|240|160|-|6|2|7.58|3.66|0.48
91|A iliaca externa sinistra 2|280|29|19|16|3|15.99|15.54|0.86
92|A uterina sinistra|260|200|-|3|2|3.19|3.19|0.33
93|A iliaca externa sinistra 3|280|230|54|15|3|15.30|15.30|0.84
94|A profunda femoris sinistra|335|30|-|8|2|7.93|7.23|0.55
95|A femoralis sinistra 1|280|51|14|15|3|15.26|14.27|0.82
96|A circumflexa femoris lateralis sinistra|240|30|-|6|3|7.69|3.62|0.48
97|A femoralis sinistra 2|280|180|56|11|1|10.78|10.78|0.68
98|A saphena sinistra|285|625|-|3|3|2.59|2.59|0.29
99|A femoralis sinistra 3|280|25|26|10|1|10.78|9.71|0.66
100|A genus descendens sinistra|260|25|-|3|1|4.46|3.47|0.38
99b|A femoralis sinistra 4|280|50|-|10|-|9.71|9.71|0.64
101|A iliaca interna dextra|330|30|10|18|6|20.19|15.12|0.92
102|A iliaca externa dextra 1|315|30|-|17|2|16.98|16.19|0.88
103|A uterina dextra|260|200|-|6|2|3.19|3.19|0.33
104|A iliaca externa dextra 2|280|29|19|16|3|15.97|15.56|0.86
105|A circumflexa ilium profunda dextra|240|160|-|3|2|7.52|3.77|0.48
106|A iliaca externa dextra 3|280|230|54|15|3|15.30|15.30|0.84
107|A profunda femoris dextra|335|30|-|8|2|7.93|7.23|0.55
108|A femoralis dextra 1|280|51|14|15|3|15.26|14.27|0.82
109|A circumflexa femoris lateralis dextra|240|30|-|6|3|7.69|3.62|0.48
110|A femoralis dextra 2|280|180|56|11|1|10.78|10.78|0.68
111|A saphena dextra|285|625|-|3|3|2.59|2.59|0.29
112|A femoralis dextra 3|280|25|26|10|1|10.78|9.71|0.66
113|A genus descendens dextra|260|25|-|3|1|4.46|3.47|0.38
112b|A femoralis dextra 4|280|50|-|10|-|9.71|9.71|0.64
"""
# Note on 58b, 69b, 99b, 112b: the paper adds these unmeasured terminal segments
# "with an artificial length of 5 cm"; Table 1 prints 5 in its mm column, so we use 50 mm.


def _parse():
    segs = {}
    for line in _TABLE1.strip().splitlines():
        f = [s.strip() for s in line.split("|")]
        segs[f[0]] = dict(
            name=f[1], angle_deg=float(f[2]), length_mm=float(f[3]),
            d_prox_mm=float(f[7]), d_dist_mm=float(f[8]), distensibility_per_mmHg=float(f[9]) * 1e-3,
        )
    return segs


SEGMENTS = _parse()

# Which segments each segment feeds (read from Fig 1 and equine anatomy).
CHILDREN = {
    "1": ["2", "3", "4"],          # aortic root: coronaries and ascending aorta
    "4": ["5", "71"],              # brachiocephalic trunk and aortic arch
    "5": ["6", "27"],
    "6": ["7", "8"], "8": ["9", "10"], "10": ["11", "12"], "12": ["13", "14"],
    "14": ["15", "16"], "16": ["17", "18"], "18": ["19", "20"], "20": ["21", "22"],
    "22": ["23", "24"], "24": ["25", "26"],                      # left fore limb
    "27": ["28", "29"], "29": ["30", "31"], "31": ["32", "33"],
    "33": ["34", "48"],            # right subclavian and bicarotid trunk
    "34": ["35", "36", "37"],
    "37": ["38", "39"], "39": ["40", "41"], "41": ["42", "43"], "43": ["44", "45"],
    "45": ["46", "47"],                                          # right fore limb
    "48": ["49", "60"],            # left and right common carotid
    "49": ["50", "51", "52"],      # equine carotid ends in three branches
    "52": ["53", "54"], "54": ["55", "56"], "56": ["57", "58"], "58": ["59", "58b"],
    "60": ["61", "62", "63"],
    "63": ["64", "65"], "65": ["66", "67"], "67": ["68", "69"], "69": ["70", "69b"],
    "71": ["72"],
    "72": ["73", "74"], "74": ["75", "76"], "76": ["77", "78"], "78": ["79", "80"],
    "80": ["81", "82"], "82": ["83", "84", "85"], "85": ["86", "87"],
    "87": ["88", "89", "101", "102"],   # aortic quadrifurcation
    "89": ["90", "91"], "91": ["92", "93"], "93": ["94", "95"], "95": ["96", "97"],
    "97": ["98", "99"], "99": ["100", "99b"],                    # left hind limb
    "102": ["103", "104"], "104": ["105", "106"], "106": ["107", "108"],
    "108": ["109", "110"], "110": ["111", "112"], "112": ["113", "112b"],  # right hind limb
}

ROOT = "1"

# Body region of each terminal branch, for comparing the cardiac-output
# distribution with the paper's Table 2. The paper does not list its grouping.
# This one is anatomical and, among the plausible groupings we tried, it best
# reproduces the paper's no-gravity column with the S1 resistances.
REGIONS = {
    "heart": ["2", "3"],                                   # coronaries
    "brain": ["50", "61", "11", "32"],                     # internal carotids, vertebrals
    "muscle": ["17", "19", "21", "23", "25", "26", "38", "40", "42", "44", "46", "47",
               "94", "96", "98", "100", "99b", "107", "109", "111", "113", "112b"],  # limbs
    "kidney": ["79", "81"],
    "splanchnic": ["75", "77", "86", "73", "88", "101"],   # gut, oesophagus, pelvic viscera
    "other": ["7", "9", "13", "15", "28", "30", "35", "36", "51", "62",
              "53", "55", "57", "59", "58b", "64", "66", "68", "70", "69b",
              "83", "84", "90", "105", "92", "103"],
}

# Locations reported in the paper's Tables 3 and 4.
LOCATIONS = {"Prox Ao": "1", "Dist Ao": "87", "CCA": "49", "MA": "26", "EIA": "93"}

# The aorta from the root to the quadrifurcation, used for pulse wave velocity
# and for the pressures along the aorta in Fig 5.
AORTA_PATH = ["1", "4", "71", "72", "74", "76", "78", "80", "82", "85", "87"]


def parent_of():
    return {c: p for p, kids in CHILDREN.items() for c in kids}


def terminals():
    return [s for s in SEGMENTS if s not in CHILDREN]


def check_tree():
    """Every segment but the root has exactly one parent; terminals match the paper."""
    kids = [c for v in CHILDREN.values() for c in v]
    assert len(kids) == len(set(kids)), "a segment has two parents"
    assert set(kids) | {ROOT} == set(SEGMENTS), set(SEGMENTS) ^ (set(kids) | {ROOT})
    assert len(SEGMENTS) == 117 and len(terminals()) == 62
    assigned = [s for v in REGIONS.values() for s in v]
    assert sorted(assigned) == sorted(terminals()), set(assigned) ^ set(terminals())
    return True
'''
FILES['horse1d/s1_table.py'] = r'''"""S1 Table of Vera et al. (2019): terminal impedance data (CC BY 4.0).

Total terminal resistance R_T (mmHg s/ml) and terminal compliance C_T (ml/mmHg)
for the 62 terminal branches. In the paper's 3-element Windkessel, R1 = Zc (the
terminal vessel's characteristic impedance) and R2 = R_T - R1.

The S1 file lists segment "79b" with the same values as 58b; that is a typo for
69b (79 is the right renal artery, listed separately, and 69b is otherwise missing).
"""

_S1 = """
2 7.22 14.36|3 7.22 14.36|7 32.80 7.97|9 29.82 0.05|11 10.98 7.53|13 32.80 11.01
15 55.44 1.34|17 55.44 0.61|19 55.44 25.80|21 55.44 0.33|23 55.44 6.34|25 67.61 1.31
26 64.46 1.65|28 32.80 3.62|30 29.82 0.05|32 10.98 3.98|35 55.44 11.00|36 55.44 3.78
38 55.44 1.46|40 55.44 25.80|42 55.44 0.33|44 61.60 7.55|46 67.61 1.31|47 64.46 1.65
50 1.10 0.48|51 8.55 0.71|53 8.55 1.50|55 8.62 0.04|57 8.92 0.10|58b 9.00 1.19
59 9.08 0.20|61 1.10 0.48|62 8.55 0.71|64 8.55 1.50|66 8.62 0.04|68 8.92 0.10
70 9.08 0.20|73 3.55 2.78|75 1.96 7.34|77 3.03 4.04|79 1.80 3.62|81 1.80 3.63
83 22.55 0.24|84 22.55 0.24|86 3.51 1.22|88 5.52 21.68|90 47.31 0.66|92 23.13 0.12
94 47.31 2.97|96 47.31 0.65|69b 9.00 1.20|98 47.31 0.07|99b 57.70 6.21|100 57.70 0.47
101 5.52 21.68|103 23.13 0.12|105 55.02 0.70|107 47.31 2.97|109 47.31 0.65|111 47.31 0.07
112b 57.70 6.21|113 52.57 0.47
"""


def load():
    """{segment: (R_T in mmHg s/ml, C_T in ml/mmHg)}; compliance is listed in 1e-2 ml/mmHg."""
    out = {}
    for item in _S1.replace("\n", "|").split("|"):
        if item.strip():
            seg, r, c = item.split()
            out[seg] = (float(r), float(c) * 1e-2)
    return out
'''
FILES['horse1d/params.py'] = r'''"""Model parameters, taken from Vera et al. (2019) unless marked as our assumption."""
from dataclasses import dataclass, field

MMHG = 133.322  # Pa per mmHg
ML = 1e-6       # m^3 per ml


@dataclass
class Params:
    # Blood (paper)
    rho: float = 1050.0          # kg/m^3
    mu: float = 0.004            # Pa s

    # Physics switches
    gravity: bool = False        # paper runs the model with and without gravity
    g: float = 9.81
    friction: str = "womersley"  # "womersley" (paper) or "poiseuille" (simpler)
    womersley_alpha_min: float = 3.0   # paper: Womersley friction where alpha > 3
    n_harmonics: int = 30              # our choice

    # Arterial wall, Eq (3) (paper)
    distensibility_scale: float = 1.0   # multiply every segment's distensibility (e.g. 0.7 = stiffer, older arteries)
    P_ref_mmHg: float = 100.0
    a1: float = 0.76
    b1: float = 5.0
    PmaxC_mmHg: float = 10.0
    Pwidth_mmHg: float = 21.0

    # Heart: time-varying elastance with internal resistance, Eqs (10)-(11) (paper)
    heart_rate_bpm: float = 40.0
    T_max: float = 0.478         # s, "systolic duration", used as time to peak elastance
    E_max: float = 0.26          # mmHg/ml
    E_min: float = 0.01          # mmHg/ml
    V0_ml: float = 0.0
    kappa: float = 55e-6         # s/ml
    P_fill_mmHg: float = 16.0    # end-diastolic pressure; filling source pressure
    R_fill: float = 0.003        # mmHg s/ml, LV filling (internal) resistance
    # Shape of the normalized elastance curve: double-Hill fit to Senzaki et al.
    # (Stergiopulos et al. 1996), stretched so the peak falls at T_max (our choice).
    hill_a1: float = 0.303
    hill_a2: float = 0.508
    hill_n1: float = 1.32
    hill_n2: float = 21.9

    # Terminal 3-element Windkessels, Eqs (7)-(8)
    terminals: str = "s1"                # "s1": the paper's S1 Table; "estimated": paper's method, our guess
    R_total_parallel: float = 0.14       # mmHg s/ml (paper: "about 0.14"); used when estimated
    terminal_compliance_fraction: float = 0.20   # paper: ~20% of total compliance
    P_out_mmHg: float = 0.0              # outflow (venous) pressure, our assumption
    # Target share of cardiac output by region. Default: the paper's own model
    # without gravity (Table 2, first column). Used only because the per-branch
    # values in the paper's S1 Table are not in the PDF.
    flow_targets: dict = field(default_factory=lambda: {
        "heart": 4.2, "brain": 15.4, "muscle": 7.7,
        "kidney": 18.2, "splanchnic": 28.3, "other": 26.2})
    murray_exponent: float = 1.5   # within a region, flow ~ (distal area)^1.5, i.e. r^3

    # Numerics (ours)
    dx_target: float = 0.01      # m
    cfl: float = 0.8
    # Shock-capturing damping (Jameson-type, our choice). The paper's nonlinear wall law
    # makes the pulse front steepen along the aorta; this keeps the explicit scheme from
    # ringing there. 0 turns it off.
    shock_capturing: float = 1.0
    n_cycles: int = 8            # paper: 8 cardiac cycles
    history_samples: int = 256   # samples per cycle for the Womersley friction history
    output_samples: int = 1000   # samples saved for the last cycle
    P_init_mmHg: float = 100.0   # paper initial condition
    Q_init_ml_s: float = 1.0     # paper initial condition

    @property
    def T(self):
        return 60.0 / self.heart_rate_bpm
'''
FILES['horse1d/womersley.py'] = r'''"""Witzig-Womersley wall shear stress for pulsatile flow in a rigid tube.

For the n-th harmonic Q_n of the flow, the wall shear stress is (paper, Eq 6)

    tau_n = mu / (pi R^3) * Q_n * F(Lambda_n),
    F(L) = L * J1(L)/J0(L) / (1 - 2 J1(L) / (L J0(L))),
    Lambda_n = alpha_n * i^(3/2),   alpha_n = R * sqrt(n * omega * rho / mu).

For slow flow (alpha -> 0) F -> -4, which is Poiseuille: tau = -4 mu Q / (pi R^3).
For fast flow (large alpha) |F| grows like alpha: a thin boundary layer at the wall.
"""
import numpy as np
from scipy.special import jve


def womersley_number(R, freq_hz, rho, mu):
    return R * np.sqrt(2.0 * np.pi * freq_hz * rho / mu)


def shear_factor(alpha):
    """F(Lambda) for real Womersley number(s) alpha >= 0 (array ok)."""
    alpha = np.asarray(alpha, dtype=float)
    out = np.full(alpha.shape, -4.0 + 0j)
    big = alpha > 1e-3
    L = alpha[big] * np.exp(1j * 3 * np.pi / 4)
    # jve is exponentially scaled, so the ratio stays finite for large alpha
    r = jve(1, L) / jve(0, L)
    out[big] = L * r / (1.0 - 2.0 * r / L)
    return out


def wall_shear_from_flow(Q_samples, R, T, rho, mu, n_harmonics):
    """Wall shear stress tau(t) (Pa) from one period of flow samples (m^3/s).

    Q_samples: array (..., M) of equally spaced samples over one period.
    R: radius (m), broadcastable to Q_samples[..., 0].
    """
    M = Q_samples.shape[-1]
    Qh = np.fft.rfft(Q_samples, axis=-1) / M
    n = np.arange(Qh.shape[-1])
    alpha = womersley_number(np.asarray(R)[..., None], n / T, rho, mu)
    F = shear_factor(alpha)
    F[..., 0] = -4.0
    Th = (mu / (np.pi * np.asarray(R)[..., None] ** 3)) * F * Qh
    Th[..., n_harmonics + 1:] = 0.0
    return np.fft.irfft(Th * M, n=M, axis=-1)
'''
FILES['horse1d/model.py'] = r'''"""1D model of the equine arterial tree, after Vera et al. (2019), PLoS ONE 14(8): e0221425.

In each of the 117 segments we solve, for pressure P(x,t) and flow Q(x,t),

    mass:      C_A(P) dP/dt + dQ/dx = 0
    momentum:  dQ/dt + d(delta Q^2/A)/dx + (A/rho) dP/dx = 2 pi R tau_w / rho - A g sin(phi)

with the paper's nonlinear wall law A(P), Womersley (or Poiseuille) wall shear
tau_w, and gravity along each segment. Boundary conditions:
  * aortic root: time-varying elastance left ventricle with internal resistance,
  * branch points: equal pressure and conservation of flow,
  * 62 terminal branches: 3-element Windkessels.

Numerics (ours; the paper used an implicit scheme in MATLAB): explicit MacCormack
in the interior, and the method of characteristics at every segment end.
"""
import time
from collections import deque

import numpy as np

from . import network as net
from .params import Params, MMHG, ML
from .womersley import womersley_number, shear_factor

_trapezoid = getattr(np, "trapezoid", None) or np.trapz   # np.trapz is np.trapezoid from NumPy 2.0 on


class HorseArteryModel:
    def __init__(self, params=None, terminal_table=None):
        self.p = params or Params()
        self.terminal_table = terminal_table   # optional {seg: (R1, R2, C)} in mmHg s/ml, ml/mmHg
        self._build_grid()
        self._build_wall_law()
        self._build_terminals()
        self._build_heart()

    # ------------------------------------------------------------------ grid
    def _build_grid(self):
        p = self.p
        order, q = [], deque([net.ROOT])
        while q:
            s = q.popleft()
            order.append(s)
            q.extend(net.CHILDREN.get(s, []))
        self.order = order
        parent = net.parent_of()

        start, end, nodes = {}, {}, 0
        x_list, Ar, Dm, sphi, dxn, zz, vid = [], [], [], [], [], [], []
        z_start = {}
        for k, s in enumerate(order):
            seg = net.SEGMENTS[s]
            L = seg["length_mm"] / 1000.0
            N = max(3, int(round(L / p.dx_target)) + 1)
            x = np.linspace(0.0, L, N)
            r = 0.5e-3 * (seg["d_prox_mm"] + (seg["d_dist_mm"] - seg["d_prox_mm"]) * x / L)
            sin_phi = np.sin(np.radians(seg["angle_deg"]))
            z0 = 0.0 if s == net.ROOT else z_start[parent[s]] + net.SEGMENTS[parent[s]]["length_mm"] / 1000.0 * np.sin(
                np.radians(net.SEGMENTS[parent[s]]["angle_deg"]))
            z_start[s] = z0
            start[s], end[s] = nodes, nodes + N - 1
            nodes += N
            x_list.append(x)
            Ar.append(np.pi * r ** 2)
            Dm.append(np.full(N, seg["distensibility_per_mmHg"] * p.distensibility_scale))
            sphi.append(np.full(N, sin_phi))
            dxn.append(np.full(N, L / (N - 1)))
            zz.append(z0 + x * sin_phi)
            vid.append(np.full(N, k))
        self.n_nodes = nodes
        self.start, self.end = start, end
        self.x = np.concatenate(x_list)
        self.Aref = np.concatenate(Ar)
        self.Rref = np.sqrt(self.Aref / np.pi)
        self.Dmm = np.concatenate(Dm)
        self.sinphi = np.concatenate(sphi)
        self.dx = np.concatenate(dxn)
        self.z = np.concatenate(zz)
        self.vessel = np.concatenate(vid)

        is_first = np.zeros(nodes, bool)
        is_last = np.zeros(nodes, bool)
        is_first[[start[s] for s in order]] = True
        is_last[[end[s] for s in order]] = True
        self.fwd = np.where(~is_last)[0]
        self.bwd = np.where(~is_first)[0]
        self.interior = np.where(~is_first & ~is_last)[0]

        # Womersley number at the heart rate, per node
        f0 = 1.0 / p.T
        self.alpha = womersley_number(self.Rref, f0, p.rho, p.mu)
        self.wom = (self.alpha > p.womersley_alpha_min) & (p.friction == "womersley")
        # Convective profile coefficient: flat (1) where inertia dominates, Poiseuille (4/3) otherwise
        self.delta = np.where(self.alpha > p.womersley_alpha_min, 1.0, 4.0 / 3.0)

        # Branch points
        jp, jd = [], []
        for j, (par, kids) in enumerate((s, net.CHILDREN[s]) for s in order if s in net.CHILDREN):
            jp.append((j, end[par]))
            for kid in kids:
                jd.append((j, start[kid]))
        self.n_junctions = len(jp)
        self.jp_id = np.array([a for a, _ in jp])
        self.jp_node = np.array([b for _, b in jp])
        self.jd_id = np.array([a for a, _ in jd])
        self.jd_node = np.array([b for _, b in jd])
        self.term_segs = [s for s in order if s not in net.CHILDREN]
        self.term_node = np.array([end[s] for s in self.term_segs])
        self.root_node = start[net.ROOT]
        self.ends = np.array([end[s] for s in order])
        self.starts = np.array([start[s] for s in order])

    # ------------------------------------------------------------- wall law
    def _build_wall_law(self):
        p = self.p
        self._Cp_ref = p.a1 + p.b1 / (1.0 + ((p.P_ref_mmHg - p.PmaxC_mmHg) / p.Pwidth_mmHg) ** 2)
        self._F_ref = self._F(p.P_ref_mmHg)

    def _F(self, Pm):
        p = self.p
        return p.a1 * Pm + p.b1 * p.Pwidth_mmHg * np.arctan((Pm - p.PmaxC_mmHg) / p.Pwidth_mmHg)

    # Below this fraction of the reference area the wall law continues as a smooth
    # exponential collapse (our addition). Normal runs never get there; it only matters
    # when pressure in the head falls near or below zero (standing horse, very slow heart).
    _F0 = 0.25

    def _wall(self, P, Aref, Dmm):
        """Area (m^2) and area compliance dA/dP (m^2/Pa) from pressure (Pa).
        Above collapse: A integrates the paper's compliance law, Eq (3)."""
        p = self.p
        Pm = P / MMHG
        f = 1.0 + Dmm * (self._F(Pm) - self._F_ref) / self._Cp_ref
        Cp = p.a1 + p.b1 / (1.0 + ((Pm - p.PmaxC_mmHg) / p.Pwidth_mmHg) ** 2)
        dfdP = Dmm * Cp / self._Cp_ref / MMHG
        F0 = self._F0
        e = np.exp(np.minimum(f - F0, 0.0) / F0)
        A = np.where(f >= F0, Aref * f, Aref * F0 * e)
        C = np.where(f >= F0, Aref * dfdP, Aref * e * dfdP)
        return A, C

    def area(self, P):
        """Lumen area (m^2) at every node from pressure (Pa)."""
        return self._wall(P, self.Aref, self.Dmm)[0]

    def compliance(self, P):
        """Area compliance C_A = dA/dP (m^2/Pa) at every node."""
        return self._wall(P, self.Aref, self.Dmm)[1]

    def area_at(self, i, P):
        """Area at node i for an array of pressures (e.g. over time)."""
        return self._wall(P, self.Aref[i], self.Dmm[i])[0]

    # ------------------------------------------------------------ terminals
    def _build_terminals(self):
        p = self.p
        tn = self.term_node
        c_ref = np.sqrt(1.0 / (p.rho * self.Dmm[tn] / MMHG))
        Zc = p.rho * c_ref / self.Aref[tn]                       # Pa s/m^3
        self.Zc_term = Zc
        R_tot = p.R_total_parallel * MMHG / ML
        self.region_of = {s: r for r, segs in net.REGIONS.items() for s in segs}
        Cnode = self.compliance(np.full(self.n_nodes, p.P_ref_mmHg * MMHG))
        self.C_vessels = self._integrate_segments(Cnode).sum()
        table = self.terminal_table
        if table is None and p.terminals == "s1":
            from .s1_table import load
            table = load()
        if table is not None:
            # Paper's S1 Table: total resistance R_T and compliance per terminal.
            # R1 = Zc (paper); where Zc > R_T (internal carotids, small head branches)
            # we use R1 = R_T, R2 = 0, i.e. a pure resistance.
            RT = np.array([table[s][0] for s in self.term_segs]) * MMHG / ML
            C = np.array([table[s][1] for s in self.term_segs]) * ML / MMHG
            R1 = np.minimum(Zc, RT)
            R2 = RT - R1
        else:
            # Flow share of each terminal: region target, split within the region by Murray's law
            frac = np.zeros(len(tn))
            total_target = sum(p.flow_targets.values())
            for r, segs in net.REGIONS.items():
                idx = [self.term_segs.index(s) for s in segs]
                w = self.Aref[tn[idx]] ** p.murray_exponent
                frac[idx] = p.flow_targets[r] / total_target * w / w.sum()
            RT = R_tot / frac                                    # parallel combination = R_tot
            R1 = np.minimum(Zc, 0.9 * RT)                        # paper: R1 = Zc
            R2 = RT - R1
            # Terminal compliance: a fixed share of total compliance, split by
            # distal area compliance (Eq 8)
            f = p.terminal_compliance_fraction
            C_T = f / (1.0 - f) * self.C_vessels
            C = C_T * Cnode[tn] / Cnode[tn].sum()
        self.R1, self.R2, self.CT = R1, R2, C
        # decay factor for the Windkessel capacitor over one step (set in run)
        self._tauW = self.R2 * self.CT

    def _integrate_segments(self, f):
        """Trapezoid integral of a node field over each segment."""
        out = []
        for s in self.order:
            i0, i1 = self.start[s], self.end[s]
            out.append(_trapezoid(f[i0:i1 + 1], self.x[i0:i1 + 1]))
        return np.array(out)

    # ---------------------------------------------------------------- heart
    def _build_heart(self):
        p = self.p
        t = np.linspace(0, 3, 30001)
        e = self._hill(t, 1.0, 1.0)
        self._t_peak_unit = t[np.argmax(e)]
        self._hill_norm = e.max()

    def _hill(self, t, scale, norm):
        p = self.p
        t1, t2 = p.hill_a1 * scale, p.hill_a2 * scale
        g1 = (t / t1) ** p.hill_n1
        return (g1 / (1 + g1)) / (1 + (t / t2) ** p.hill_n2) / norm

    def elastance(self, t_in_cycle):
        """Isovolumic elastance E*(t), Pa/m^3."""
        p = self.p
        scale = p.T_max / self._t_peak_unit
        e = self._hill(t_in_cycle, scale, self._hill_norm)
        return (p.E_min + (p.E_max - p.E_min) * e) * MMHG / ML

    # ------------------------------------------------------------ physics
    def source(self, P, Q, A, tau_lag=None, Q_lag=None):
        """Friction + gravity per unit length, divided by rho (m^3/s^2 per m)."""
        p = self.p
        S = -8.0 * np.pi * p.mu / p.rho * Q / A                 # Poiseuille
        if tau_lag is not None:
            w = self.wom
            R = self.Rref[w]
            tau = tau_lag[w] - 4.0 * p.mu / (np.pi * R ** 3) * (Q[w] - Q_lag[w])
            S[w] = 2.0 * np.pi * R * tau / p.rho
        if p.gravity:
            S = S - A * p.g * self.sinphi
        return S

    # ---------------------------------------------------------------- run
    def run(self, n_cycles=None, verbose=True, progress=None):
        """Run n_cycles heartbeats. progress(cycle, n_cycles) is called after each beat, if given."""
        p = self.p
        n_cycles = n_cycles or p.n_cycles
        T, rho = p.T, p.rho
        N = self.n_nodes

        # initial state (paper: 100 mmHg, 1 ml/s); with gravity start from hydrostatic balance
        P = np.full(N, p.P_init_mmHg * MMHG)
        if p.gravity:
            P = P - rho * p.g * self.z
        Q = np.full(N, p.Q_init_ml_s * ML)
        Pc = P[self.term_node] - self.R1 * Q[self.term_node]
        V = p.P_fill_mmHg / p.E_min * ML + p.V0_ml * ML
        V0 = p.V0_ml * ML
        P_fill = p.P_fill_mmHg * MMHG
        R_fill = p.R_fill * MMHG / ML
        kappa = p.kappa / ML                                       # s/m^3
        P_out = p.P_out_mmHg * MMHG

        # time step from the fastest wave at high pressure
        Phi = np.full(N, 250 * MMHG)
        c_max = np.sqrt(self.area(Phi) / (rho * self.compliance(Phi)))
        dt = p.cfl * np.min(self.dx / (c_max + 2.0))
        steps_per_cycle = int(np.ceil(T / dt))
        dt = T / steps_per_cycle
        self.dt = dt
        with np.errstate(divide="ignore"):
            decay = np.where(self._tauW > 0, np.exp(-dt / np.maximum(self._tauW, 1e-300)), 0.0)

        M = p.history_samples
        hist_every = steps_per_cycle / M
        Q_hist = np.zeros((N, M))
        tau_lag = Q_lag = None
        tau_tab = Q_tab = None

        n_out = p.output_samples
        out_every = max(1, steps_per_cycle // n_out)
        n_out = steps_per_cycle // out_every
        rec = dict(t=np.zeros(n_out), P=np.zeros((N, n_out)), Q=np.zeros((N, n_out)),
                   Plv=np.zeros(n_out), V=np.zeros(n_out), Qroot=np.zeros(n_out), Pc=np.zeros((len(Pc), n_out)))
        root_cycles = []

        wtab = self.wom.copy()
        Rw = self.Rref[wtab]
        harm = np.arange(M // 2 + 1)
        Fw = shear_factor(womersley_number(Rw[:, None], harm[None, :] / T, rho, p.mu))
        Fw[:, 0] = -4.0
        Fw[:, p.n_harmonics + 1:] = 0.0

        t = 0.0
        t_wall = time.time()
        fwd, bwd, inner = self.fwd, self.bwd, self.interior
        root = self.root_node
        ends, starts = self.ends, self.starts
        for cyc in range(n_cycles):
            root_P = np.zeros(M)
            last = cyc == n_cycles - 1
            for k in range(steps_per_cycle):
                tc = k * dt
                if tau_tab is not None:
                    ph = k / hist_every
                    m0 = int(ph) % M
                    wph = ph - int(ph)
                    m1 = (m0 + 1) % M
                    tau_lag = (1 - wph) * tau_tab[:, m0] + wph * tau_tab[:, m1]
                    Q_lag = (1 - wph) * Q_tab[:, m0] + wph * Q_tab[:, m1]

                A = self.area(P)
                CA = self.compliance(P)
                c = np.sqrt(A / (rho * CA))
                Z = rho * c / A
                S = self.source(P, Q, A, tau_lag, Q_lag)
                G = self.delta * Q * Q / A

                # --- MacCormack in the interior (alternate the sweep direction)
                Ps, Qs = P.copy(), Q.copy()
                if k % 2 == 0:
                    i, j = fwd, fwd + 1
                    Ps[i] = P[i] - dt * (Q[j] - Q[i]) / self.dx[i] / CA[i]
                    Qs[i] = Q[i] - dt * ((G[j] - G[i]) / self.dx[i] + A[i] / rho * (P[j] - P[i]) / self.dx[i]) + dt * S[i]
                    a, b = inner - 1, inner
                else:
                    i, j = bwd - 1, bwd
                    Ps[j] = P[j] - dt * (Q[j] - Q[i]) / self.dx[j] / CA[j]
                    Qs[j] = Q[j] - dt * ((G[j] - G[i]) / self.dx[j] + A[j] / rho * (P[j] - P[i]) / self.dx[j]) + dt * S[j]
                    a, b = inner, inner + 1
                As = self.area(Ps)
                CAs = self.compliance(Ps)
                Ss = self.source(Ps, Qs, As, tau_lag, Q_lag)
                Gs = self.delta * Qs * Qs / As
                Pn, Qn = P.copy(), Q.copy()
                dxi = self.dx[inner]
                Pn[inner] = 0.5 * (P[inner] + Ps[inner] - dt * (Qs[b] - Qs[a]) / dxi / CAs[inner])
                Qn[inner] = 0.5 * (Q[inner] + Qs[inner]
                                   - dt * ((Gs[b] - Gs[a]) / dxi + As[inner] / rho * (Ps[b] - Ps[a]) / dxi)
                                   + dt * Ss[inner])

                # --- characteristics arriving at every segment end (from time level n)
                lam = np.clip((Q[ends] / A[ends] + c[ends]) * dt / self.dx[ends], 0, 1)
                Pf = P[ends] - lam * (P[ends] - P[ends - 1])
                Qf = Q[ends] - lam * (Q[ends] - Q[ends - 1])
                Wp = Pf + Z[ends] * Qf + Z[ends] * S[ends] * dt          # P + Z Q along C+
                lam = np.clip((c[starts] - Q[starts] / A[starts]) * dt / self.dx[starts], 0, 1)
                Pf = P[starts] + lam * (P[starts + 1] - P[starts])
                Qf = Q[starts] + lam * (Q[starts + 1] - Q[starts])
                Wm = Pf - Z[starts] * Qf - Z[starts] * S[starts] * dt    # P - Z Q along C-
                W_end = np.zeros(N); W_end[ends] = Wp
                W_start = np.zeros(N); W_start[starts] = Wm

                # --- branch points: same pressure, flow conserved
                jp, jd = self.jp_node, self.jd_node
                num = np.bincount(self.jp_id, W_end[jp] / Z[jp], self.n_junctions) + \
                    np.bincount(self.jd_id, W_start[jd] / Z[jd], self.n_junctions)
                den = np.bincount(self.jp_id, 1 / Z[jp], self.n_junctions) + \
                    np.bincount(self.jd_id, 1 / Z[jd], self.n_junctions)
                Pj = num / den
                Pn[jp] = Pj[self.jp_id]
                Qn[jp] = (W_end[jp] - Pn[jp]) / Z[jp]
                Pn[jd] = Pj[self.jd_id]
                Qn[jd] = (Pn[jd] - W_start[jd]) / Z[jd]

                # --- terminal Windkessels
                tn = self.term_node
                Qt = (W_end[tn] - Pc) / (Z[tn] + self.R1)
                Pn[tn] = Pc + self.R1 * Qt
                Qn[tn] = Qt
                # capacitor: C dPc/dt = Q - (Pc - P_out)/R2, integrated exactly over dt
                Pinf = P_out + self.R2 * Qt
                Pc = Pinf + (Pc - Pinf) * decay

                # --- left ventricle at the aortic root
                Es = self.elastance(tc)
                Wr = W_start[root]
                Zr = Z[root]
                q = (Es * (V - V0) - Wr) / (Es * (V - V0) * kappa + Zr)
                if q > 0:
                    Qn[root] = q
                    Pn[root] = Wr + Zr * q
                    Plv = Pn[root]
                    V -= q * dt
                else:
                    Qn[root] = 0.0
                    Pn[root] = Wr
                    Plv = Es * (V - V0)
                    if Plv < P_fill:
                        V += (P_fill - Plv) / R_fill * dt

                # --- shock-capturing damping where the pressure has a sharp kink
                if p.shock_capturing > 0:
                    nu = np.zeros(N)
                    num = np.abs(Pn[inner + 1] - 2 * Pn[inner] + Pn[inner - 1])
                    nu[inner] = p.shock_capturing * num / (Pn[inner + 1] + 2 * Pn[inner] + Pn[inner - 1])
                    eR = np.minimum(np.maximum(nu[inner], nu[inner + 1]), 0.25)
                    eL = np.minimum(np.maximum(nu[inner - 1], nu[inner]), 0.25)
                    for U in (Pn, Qn):
                        U[inner] = U[inner] + eR * (U[inner + 1] - U[inner]) - eL * (U[inner] - U[inner - 1])

                P, Q = Pn, Qn
                t += dt

                # --- records
                if k % hist_every < 1.0:
                    m = int(k / hist_every)
                    if m < M:
                        Q_hist[:, m] = Q
                        root_P[m] = P[root]
                if last and k % out_every == 0 and k // out_every < n_out:
                    o = k // out_every
                    rec["t"][o] = tc
                    rec["P"][:, o] = P
                    rec["Q"][:, o] = Q
                    rec["Plv"][o] = Plv
                    rec["V"][o] = V
                    rec["Qroot"][o] = Q[root]
                    rec["Pc"][:, o] = Pc
                if k % 50 == 0 and not np.isfinite(P).all():
                    bad = np.where(~np.isfinite(P))[0]
                    segs = sorted({self.order[self.vessel[b]] for b in bad})
                    raise FloatingPointError(f"solution blew up at t={t:.3f} s in segments {segs[:8]}")

            # end of cycle: Womersley wall shear for the next cycle, from this cycle's flow
            root_cycles.append(root_P / MMHG)
            if p.friction == "womersley":
                Qh = np.fft.rfft(Q_hist[wtab], axis=1) / M
                Th = (p.mu / (np.pi * Rw[:, None] ** 3)) * Fw * Qh
                tau_w = np.fft.irfft(Th * M, n=M, axis=1)
                tau_tab = np.zeros((N, M)); tau_tab[wtab] = tau_w
                Q_tab = Q_hist.copy()
            if progress is not None:
                progress(cyc + 1, n_cycles)
            if verbose:
                d = np.abs(root_cycles[-1] - root_cycles[-2]).max() if len(root_cycles) > 1 else np.nan
                print(f"cycle {cyc + 1}/{n_cycles}: root P {root_P.min() / MMHG:5.1f}-{root_P.max() / MMHG:5.1f} mmHg, "
                      f"change vs last cycle {d:5.2f} mmHg, {time.time() - t_wall:5.1f} s")

        self.rec = rec
        self.root_cycles = root_cycles
        self.final_state = dict(P=P, Q=Q, Pc=Pc, V=V)
        return rec

    # ----------------------------------------------------------- helpers
    def node(self, seg, where="mid"):
        s, e = self.start[seg], self.end[seg]
        return {"start": s, "end": e, "mid": (s + e) // 2}[where]

    def node_at_distance(self, path, dist_m):
        """Node at a given distance from the start of a path of segments."""
        acc = 0.0
        for s in path:
            L = net.SEGMENTS[s]["length_mm"] / 1000.0
            if dist_m <= acc + L:
                i = self.start[s] + int(round((dist_m - acc) / L * (self.end[s] - self.start[s])))
                return i
            acc += L
        return self.end[path[-1]]
'''
FILES['horse1d/analysis.py'] = r'''"""Numbers the paper reports, computed from a finished model run (the last cycle)."""
import numpy as np

from . import network as net
from .params import MMHG, ML
from .womersley import wall_shear_from_flow


def global_metrics(m):
    r, p = m.rec, m.p
    dt = r["t"][1] - r["t"][0]
    SV = r["Qroot"].sum() * dt / ML
    EDV, ESV = r["V"].max() / ML, r["V"].min() / ML
    P = r["P"][m.node("1", "mid")] / MMHG
    CO = SV * 60.0 / p.T / 1000.0
    return dict(CO_L_min=CO, SV_ml=SV, EF_pct=100 * SV / EDV, EDV_ml=EDV, ESV_ml=ESV,
                SBP=P.max(), DBP=P.min(), MAP=P.mean(), PP=P.max() - P.min(),
                TVR=P.mean() / (CO * 1000 / 60.0))


def co_distribution(m):
    """Share of cardiac output (%) by body region, from mean terminal flows."""
    r = m.rec
    qmean = r["Q"][m.term_node].mean(axis=1)
    total = qmean.sum()
    out = {}
    for region, segs in net.REGIONS.items():
        idx = [m.term_segs.index(s) for s in segs]
        out[region] = 100 * qmean[idx].sum() / total
    return out


def location_metrics(m, seg, where="mid"):
    r, p = m.rec, m.p
    i = m.node(seg, where)
    P = r["P"][i]
    Q = r["Q"][i]
    A = _area_at(m, i, P)
    U = Q / A
    Pm = P / MMHG
    # wall shear stress over the cycle
    R = m.Rref[i]
    if m.wom[i]:
        tau = wall_shear_from_flow(Q, R, p.T, p.rho, p.mu, p.n_harmonics)
    else:
        tau = -4 * p.mu * Q / (np.pi * R ** 3)
    # mean convective acceleration d/dx(delta Q^2/A), central difference
    j0, j1 = max(i - 1, m.start[seg]), min(i + 1, m.end[seg])
    G0 = m.delta[j0] * r["Q"][j0] ** 2 / _area_at(m, j0, r["P"][j0])
    G1 = m.delta[j1] * r["Q"][j1] ** 2 / _area_at(m, j1, r["P"][j1])
    conv = ((G1 - G0) / (m.x[j1] - m.x[j0])).mean()
    D = 2 * np.sqrt(A.mean() / np.pi)
    return dict(alpha=m.alpha[i], peak_U=U.max(), mean_U=U.mean(), SBP=Pm.max(), DBP=Pm.min(),
                MAP=Pm.mean(), PP=Pm.max() - Pm.min(), tau_max=np.abs(tau).max(),
                conv_mean=conv, Re=p.rho * abs(U.mean()) * D / p.mu, height_m=m.z[i])


def _area_at(m, i, P):
    return m.area_at(i, P)


def foot_time(t, P):
    """Foot of the pressure wave: maximum of the second derivative during the upstroke."""
    d1 = np.gradient(P, t)
    d2 = np.gradient(d1, t)
    k_up = np.argmax(d1)                      # steepest upstroke
    lo = max(0, k_up - int(0.15 / (t[1] - t[0])))
    k = lo + np.argmax(d2[lo:k_up + 1])
    return t[k]


def aortic_pwv(m, distances_cm=(26.4, 36.4, 46.4, 56.4, 66.4, 76.4, 86.4)):
    r = m.rec
    times = []
    for dcm in distances_cm:
        i = m.node_at_distance(net.AORTA_PATH, dcm / 100.0)
        P = _smooth(r["P"][i])
        times.append(foot_time(r["t"], P))
    times = np.array(times)
    d = np.array(distances_cm) / 100.0
    slope = np.polyfit(times, d, 1)[0]
    return slope, times


def aorta_pressures(m, distances_cm=(26.4, 36.4, 46.4, 56.4, 66.4, 76.4, 86.4)):
    r = m.rec
    return {dcm: r["P"][m.node_at_distance(net.AORTA_PATH, dcm / 100.0)] / MMHG for dcm in distances_cm}


def wave_power(m, seg, where="mid"):
    """Forward and backward wave power, d(pi)+ and d(pi)- (Mynard & Smolich 2016)."""
    r, p = m.rec, m.p
    i = m.node(seg, where)
    P, Q = _smooth(r["P"][i], 9), _smooth(r["Q"][i], 9)   # ~13 ms smoothing, so steep fronts don't swamp the plot
    A = _area_at(m, i, P)
    C = m.Aref[i] * m.Dmm[i] / MMHG       # reference area compliance
    c = np.sqrt(A.mean() / (p.rho * C))
    Zc = p.rho * c / A.mean()
    dP, dQ = np.diff(P), np.diff(Q)
    fwd = (dP + Zc * dQ) ** 2 / (4 * Zc)
    bwd = -(dP - Zc * dQ) ** 2 / (4 * Zc)
    return r["t"][1:], fwd, bwd


def _smooth(y, k=5):
    ker = np.ones(k) / k
    ypad = np.concatenate([y[-k:], y, y[:k]])
    return np.convolve(ypad, ker, mode="same")[k:-k]
'''
FILES['horse1d/paper.py'] = r'''"""Published results from Vera et al. (2019), for comparison. NA = not reported."""

GLOBAL = {  # Results, "General physiological parameters"
    "no_gravity": dict(CO_L_min=33, EF_pct=65, SV_ml=820, SBP=114, DBP=70, PP=44, MAP=93, TVR=0.17),
    "gravity": dict(CO_L_min=30, EF_pct=59, SV_ml=740, SBP=131, DBP=88, PP=43, MAP=111, TVR=0.22),
}

PWV = {"no_gravity": 5.3, "gravity": 5.2, "in_vivo": 5.27}   # m/s, aorta

CO_DISTRIBUTION = {  # Table 2, % of cardiac output
    "no_gravity": dict(heart=4.2, brain=15.4, muscle=7.7, kidney=18.2, splanchnic=28.3, other=26.2),
    "gravity": dict(heart=5.4, brain=10.5, muscle=11.3, kidney=20.7, splanchnic=32.6, other=19.5),
    "reference": dict(heart=5, brain=10, muscle=15, kidney=20, splanchnic=30, other=20),
}

LOCATION_ORDER = ["Prox Ao", "Dist Ao", "CCA", "MA", "EIA"]

TABLE3 = {  # Womersley number, max wall shear (Pa), mean convective acceleration (1e-5 m^3/s^2), Reynolds
    "alpha": [35.47, 15.05, 6.02, 3.78, 8.02],
    "tau_max": {"gravity": [2.72, 2.89, 2.28, 1.74, 1.50], "no_gravity": [3.05, 3.11, 2.09, 1.79, 1.70]},
    "conv_mean": {"gravity": [57.32, -27.75, -0.12, 0.27, -1.02], "no_gravity": [68.67, -31.55, 6.79, 0.06, -1.32]},
    "Re": {"gravity": [2404, 870, 1160, 98, 278], "no_gravity": [2660, 749, 2312, 68, 195]},
}

TABLE4 = {  # model values and in vivo measurements
    "peak_U": {"in_vivo": [0.742, None, 0.846, 0.403, 0.457],
               "no_gravity": [0.955, 0.834, 1.544, 0.464, 0.642],
               "gravity": [0.731, 0.765, 1.253, 0.395, 0.560]},
    "MAP": {"in_vivo": [104.5, 119.7, 116.9, None, None],
            "no_gravity": [92.82, 94.63, 88.61, 93.54, 94.77],
            "gravity": [110.74, 95.74, 64.04, 132.34, 105.54]},
    "SBP": {"in_vivo": [116.7, 134.4, 134.8, None, None],
            "no_gravity": [113.62, 160.76, 145.08, 160.68, 174.93],
            "gravity": [131.11, 159.78, 103.44, 190.17, 175.73]},
    "DBP": {"in_vivo": [92.2, 109.8, 101.9, None, None],
            "no_gravity": [69.50, 66.70, 61.00, 67.70, 66.38],
            "gravity": [87.75, 69.30, 44.63, 103.78, 78.61]},
    "PP": {"in_vivo": [24.0, 24.2, 32.8, None, None],
           "no_gravity": [44.12, 94.06, 84.08, 92.97, 108.55],
           "gravity": [43.36, 90.47, 58.81, 86.39, 97.13]},
}
# In vivo: peak velocities and carotid pressures in standing horses (mean of 5);
# aortic pressures in one anaesthetized horse lying on its back.
'''
FILES['horse1d/lab.py'] = r'''"""Helpers for the Horse Artery Lab notebook: run the model with three controls, and plot.

Controls:
  standing    - include gravity (a standing horse) or not (as if lying flat)
  heart_rate  - beats per minute (the paper uses 40, a resting horse)
  stiffness   - 1 = normal arteries; 1.5 = every artery 1.5 times stiffer (an older horse)
"""
import functools
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

from . import analysis as an, network as net
from .model import HorseArteryModel
from .params import Params, MMHG

# The five places the paper reports, with plain names
SITES = {
    "Aorta by the heart": "1",
    "Aorta at the hips": "87",
    "Carotid (neck)": "49",
    "Median (front leg)": "26",
    "Ext. iliac (hind leg)": "93",
}
AORTA_CM = (26.4, 36.4, 46.4, 56.4, 66.4, 76.4, 86.4)


def make_params(standing=False, heart_rate=40.0, stiffness=1.0, cycles=8):
    # Systole shortens with the square root of the heart period (our assumption for other rates).
    return Params(gravity=bool(standing), heart_rate_bpm=float(heart_rate),
                  T_max=0.478 * (40.0 / heart_rate) ** 0.5, E_min=0.0127,
                  distensibility_scale=1.0 / float(stiffness), n_cycles=int(cycles))


BREAKDOWN = """
The model broke down for this setting ({what}).

In a standing horse the head sits about a meter above the heart, so blood pressure
in the head is about 75 mmHg lower than at the heart. With a slow heart (a long
pause between beats) or stiff arteries, pressure in the head falls close to zero
before the next beat. The head arteries then go soft and the pressure wave slows
down until the blood moves faster than the wave. The 1D equations cannot handle
that, and the numbers blow up.

Real horses avoid this: reflexes tighten blood vessels and speed up the heart, and
the veins draining the head help pull blood through. This model has neither.
Try a faster heart rate, softer arteries, or a lying horse.
"""


@functools.lru_cache(maxsize=32)
def run_horse(standing=False, heart_rate=40.0, stiffness=1.0, cycles=8, verbose=True):
    """Run the model and return a small dictionary of results (cached, so repeats are instant).
    Returns None, with an explanation, if the model breaks down."""
    m = HorseArteryModel(make_params(standing, heart_rate, stiffness, cycles))
    t0 = time.time()

    def tick(c, n):
        if verbose:
            print(f"{c}" + (" " if c < n else f"  done in {time.time() - t0:.0f} s\n"), end="", flush=True)

    if verbose:
        print(f"Running {label(dict(settings=dict(standing=standing, heart_rate=heart_rate, stiffness=stiffness)))}: "
              f"heartbeat ", end="", flush=True)
    try:
        with np.errstate(all="ignore"):
            m.run(verbose=False, progress=tick)
    except FloatingPointError:
        what = label(dict(settings=dict(standing=standing, heart_rate=heart_rate, stiffness=stiffness)))
        print(BREAKDOWN.format(what=what))
        return None
    g = an.global_metrics(m)
    sites = {}
    for name, seg in SITES.items():
        i = m.node(seg)
        P = m.rec["P"][i]
        L = an.location_metrics(m, seg)
        sites[name] = dict(seg=seg, P=P / MMHG, U=m.rec["Q"][i] / an._area_at(m, i, P),
                           SBP=L["SBP"], DBP=L["DBP"], MAP=L["MAP"], PP=L["PP"],
                           peak_U=L["peak_U"], height_m=m.z[i])
    pwv, feet = an.aortic_pwv(m, AORTA_CM)
    seg_P = {}
    for s in m.order:
        P = m.rec["P"][m.node(s)] / MMHG
        seg_P[s] = (P.mean(), P.max(), P.min())
    return dict(settings=dict(standing=bool(standing), heart_rate=float(heart_rate), stiffness=float(stiffness)),
                t=m.rec["t"].copy(), glob=g, co=an.co_distribution(m), sites=sites,
                aorta={d: P for d, P in an.aorta_pressures(m, AORTA_CM).items()},
                pwv=pwv, foot_times=dict(zip(AORTA_CM, feet)), seg_P=seg_P,
                Plv=m.rec["Plv"] / MMHG, V=m.rec["V"] * 1e6)


def label(res):
    s = res["settings"]
    txt = f"{'standing' if s['standing'] else 'lying'}, {s['heart_rate']:.0f} bpm"
    if s["stiffness"] != 1.0:
        txt += f", stiffness x{s['stiffness']:g}"
    return txt


# ------------------------------------------------------------------ tables
def summary(*results):
    """Print the key numbers for one or more runs side by side."""
    results = [r for r in results if r is not None]
    if not results:
        return
    cols = [label(r) for r in results]
    w = max(26, max(len(c) for c in cols) + 2)
    rows = [("Cardiac output (L/min)", lambda r: f"{r['glob']['CO_L_min']:.1f}"),
            ("Stroke volume (ml)", lambda r: f"{r['glob']['SV_ml']:.0f}"),
            ("Aortic pressure (mmHg)", lambda r: f"{r['glob']['SBP']:.0f}/{r['glob']['DBP']:.0f}"),
            ("Aortic pulse pressure (mmHg)", lambda r: f"{r['glob']['PP']:.0f}"),
            ("Aortic pulse wave speed (m/s)", lambda r: f"{r['pwv']:.1f}"),
            ("Share of blood to the brain (%)", lambda r: f"{r['co']['brain']:.1f}"),
            ("Share of blood to the limbs (%)", lambda r: f"{r['co']['muscle']:.1f}")]
    for name in SITES:
        rows.append((f"{name}: mean (mmHg)", lambda r, n=name: f"{r['sites'][n]['MAP']:.0f}"))
        rows.append((f"{name}: sys/dia (mmHg)", lambda r, n=name: f"{r['sites'][n]['SBP']:.0f}/{r['sites'][n]['DBP']:.0f}"))
        rows.append((f"{name}: peak speed (m/s)", lambda r, n=name: f"{r['sites'][n]['peak_U']:.2f}"))
    print(" " * 38 + "".join(f"{c:>{w}}" for c in cols))
    for lab, f in rows:
        print(f"{lab:38s}" + "".join(f"{f(r):>{w}}" for r in results))


def compare_with_paper():
    """Our resting horse, lying and standing, next to the numbers Vera et al. (2019) report."""
    from . import paper
    lying, standing = run_horse(False, 40, 1.0), run_horse(True, 40, 1.0)
    if lying is None or standing is None:
        return
    G0, G1 = paper.GLOBAL["no_gravity"], paper.GLOBAL["gravity"]
    rows = [("Cardiac output (L/min)", f"{G0['CO_L_min']}", f"{lying['glob']['CO_L_min']:.1f}",
             f"{G1['CO_L_min']}", f"{standing['glob']['CO_L_min']:.1f}"),
            ("Stroke volume (ml)", f"{G0['SV_ml']}", f"{lying['glob']['SV_ml']:.0f}",
             f"{G1['SV_ml']}", f"{standing['glob']['SV_ml']:.0f}"),
            ("Aortic pressure (mmHg)", f"{G0['SBP']}/{G0['DBP']}", f"{lying['glob']['SBP']:.0f}/{lying['glob']['DBP']:.0f}",
             f"{G1['SBP']}/{G1['DBP']}", f"{standing['glob']['SBP']:.0f}/{standing['glob']['DBP']:.0f}"),
            ("Aortic pulse wave speed (m/s)", f"{paper.PWV['no_gravity']}", f"{lying['pwv']:.1f}",
             f"{paper.PWV['gravity']}", f"{standing['pwv']:.1f}")]
    for n, name in enumerate(SITES):
        for key, lab_txt, nd in (("MAP", "mean pressure (mmHg)", 0), ("peak_U", "peak speed (m/s)", 2)):
            rows.append((f"{name}: {lab_txt}", f"{paper.TABLE4[key]['no_gravity'][n]:.{nd}f}",
                         f"{lying['sites'][name][key]:.{nd}f}", f"{paper.TABLE4[key]['gravity'][n]:.{nd}f}",
                         f"{standing['sites'][name][key]:.{nd}f}"))
    print(f"{'':44s}{'Study, lying':>14s}{'Ours, lying':>13s}{'Study, standing':>17s}{'Ours, standing':>16s}")
    for r in rows:
        print(f"{r[0]:44s}{r[1]:>14s}{r[2]:>13s}{r[3]:>17s}{r[4]:>16s}")
    print("\nMeasured in standing horses: aortic pulse wave speed 5.27 m/s; peak blood speed "
          "0.74 m/s (aorta by the heart), 0.85 (carotid), 0.40 (median), 0.46 (external iliac).")


# ------------------------------------------------------------------ plots
def plot_sites(*results, labels=None):
    """Blood speed and pressure over one heartbeat at the five sites."""
    results = [r for r in results if r is not None]
    if not results:
        return None
    fig, ax = plt.subplots(2, 5, figsize=(17, 6), sharex=True)
    styles = ["k", "tab:red", "tab:blue", "tab:green"]
    for r, col in zip(results, styles):
        for j, name in enumerate(SITES):
            s = r["sites"][name]
            ax[0, j].plot(r["t"], s["U"], color=col, lw=2, label=label(r))
            ax[1, j].plot(r["t"], s["P"], color=col, lw=2, label=label(r))
    for j, name in enumerate(SITES):
        ax[0, j].set_title(name)
        ax[0, j].axhline(0, color="0.6", lw=0.6)
        ax[1, j].set_xlabel("Time (s)")
    ax[0, 0].set_ylabel("Blood speed (m/s)")
    ax[1, 0].set_ylabel("Pressure (mmHg)")
    ax[0, 0].legend(frameon=False, fontsize=9)
    fig.tight_layout()
    return fig


def tree_coordinates():
    """(x, y) of the start and end of every segment, in meters; the heart is at (0, 0).
    x points toward the tail, y up; the horse faces left."""
    par = net.parent_of()
    ends = {}

    def end(s):
        if s in ends:
            return ends[s]
        x0, y0 = (0.0, 0.0) if s == net.ROOT else end(par[s])[1]
        a = np.radians(net.SEGMENTS[s]["angle_deg"])
        L = net.SEGMENTS[s]["length_mm"] / 1000.0
        ends[s] = ((x0, y0), (x0 + L * np.cos(a), y0 + L * np.sin(a)))
        return ends[s]

    for s in net.SEGMENTS:
        end(s)
    return ends


def plot_tree(ax=None, show_sites=True):
    """The 117 arteries drawn from the paper's lengths and angles, with the five sites marked."""
    xy = tree_coordinates()
    segs = [[a, b] for a, b in xy.values()]
    widths = [0.8 + 7.0 * ((0.5 * (net.SEGMENTS[s]["d_prox_mm"] + net.SEGMENTS[s]["d_dist_mm"])) / 68.0) ** 0.7
              for s in xy]
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6.5))
    ax.add_collection(LineCollection(segs, colors="0.35", linewidths=widths, capstyle="round"))
    ax.set_xlim(-0.75, 1.3)
    ax.set_ylim(-0.95, 1.08)
    ax.set_aspect("equal")
    ax.set_xlabel("Toward the tail (m)")
    ax.set_ylabel("Height above the heart (m)")
    if show_sites:
        offsets = {"1": (0.2, -0.17, "left"), "87": (-0.1, 0.12, "center"), "49": (0.14, 0.05, "center"),
                   "26": (-0.05, -0.18, "center"), "93": (0.07, -0.2, "left")}
        for name, seg in SITES.items():
            (x0, y0), (x1, y1) = xy[seg]
            x, y = 0.5 * (x0 + x1), 0.5 * (y0 + y1)
            ax.plot(x, y, "o", color="tab:red", ms=9, zorder=5)
            dx, dy, ha = offsets[seg]
            ax.annotate(name.replace(" (", "\n("), (x, y), (x + dx, y + dy), fontsize=9.5, color="tab:red",
                        arrowprops=dict(arrowstyle="-", color="tab:red", lw=0.8), ha=ha)
    ax.set_title("The horse's arteries in the model (horse facing left)")
    return ax


def plot_map(res, quantity="mean", ax=None, vmin=None, vmax=None, title=None, colorbar=True):
    """The arterial tree drawn from the paper's lengths and angles, colored by pressure.
    quantity: "mean", "systolic", or "pulse" (systolic minus diastolic)."""
    xy = tree_coordinates()
    segs, vals, widths = [], [], []
    for s, (a, b) in xy.items():
        mean, sys, dia = res["seg_P"][s]
        v = {"mean": mean, "systolic": sys, "pulse": sys - dia}[quantity]
        segs.append([a, b])
        vals.append(v)
        d = 0.5 * (net.SEGMENTS[s]["d_prox_mm"] + net.SEGMENTS[s]["d_dist_mm"])
        widths.append(0.8 + 7.0 * (d / 68.0) ** 0.7)
    if ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
    lc = LineCollection(segs, array=np.array(vals), cmap="viridis", linewidths=widths, capstyle="round")
    lc.set_clim(vmin if vmin is not None else min(vals), vmax if vmax is not None else max(vals))
    ax.add_collection(lc)
    ax.set_xlim(-0.75, 1.3)
    ax.set_ylim(-0.95, 1.08)
    ax.set_aspect("equal")
    ax.set_xlabel("Toward the tail (m)")
    ax.set_ylabel("Height above the heart (m)")
    ax.set_title(title or f"{quantity.capitalize()} pressure, {label(res)}")
    for txt, (x, y), ha in [("head", (-0.6, 0.98), "right"), ("heart", (0.06, -0.1), "left"),
                            ("front leg", (-0.1, -0.36), "right"), ("hind leg", (1.0, -0.5), "right"),
                            ("aorta", (0.5, 0.26), "center"), ("neck", (-0.36, 0.5), "right")]:
        ax.annotate(txt, (x, y), fontsize=9, color="0.25", ha=ha)
    if colorbar:
        plt.colorbar(lc, ax=ax, label={"mean": "Mean", "systolic": "Systolic", "pulse": "Pulse"}[quantity] + " pressure (mmHg)",
                     shrink=0.8)
    return ax


def plot_maps(*results, quantity="mean"):
    """Pressure maps side by side, on one color scale."""
    results = [r for r in results if r is not None]
    if not results:
        return None
    allv = []
    for r in results:
        for mean, sys, dia in r["seg_P"].values():
            allv.append({"mean": mean, "systolic": sys, "pulse": sys - dia}[quantity])
    fig, axs = plt.subplots(1, len(results), figsize=(6.4 * len(results) + 0.8, 5.4))
    axs = np.atleast_1d(axs)
    for r, a in zip(results, axs):
        plot_map(r, quantity, ax=a, vmin=min(allv), vmax=max(allv), colorbar=(a is axs[-1]))
    fig.tight_layout()
    return fig


def plot_aorta(*results):
    """Pressure along the aorta (distances from the heart) and the arrival time of the pulse."""
    results = [r for r in results if r is not None]
    if not results:
        return None
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    styles = ["k", "tab:red", "tab:blue", "tab:green"]
    for r, col in zip(results, styles):
        d = np.array(AORTA_CM)
        ax[0].plot(d, [r["aorta"][x].max() for x in AORTA_CM], "o-", color=col, label=f"systolic, {label(r)}")
        ax[0].plot(d, [r["aorta"][x].mean() for x in AORTA_CM], "s--", color=col, label=f"mean, {label(r)}")
        ax[1].plot([r["foot_times"][x] * 1000 for x in AORTA_CM], d, "o-", color=col,
                   label=f"{label(r)}: {r['pwv']:.1f} m/s")
    ax[0].set_xlabel("Distance from the heart along the aorta (cm)")
    ax[0].set_ylabel("Pressure (mmHg)")
    ax[0].legend(frameon=False, fontsize=8)
    ax[0].set_title("Pressure along the aorta")
    ax[1].set_xlabel("Time the pulse arrives (ms after the beat starts)")
    ax[1].set_ylabel("Distance from the heart (cm)")
    ax[1].set_title("Pulse wave speed = slope")
    ax[1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    return fig
'''
for path, text in FILES.items():
    with open(path, 'w', encoding='utf-8') as fh:
        fh.write(text)
import importlib, sys
for name in [m for m in sys.modules if m.startswith('horse1d')]:
    del sys.modules[name]
import matplotlib.pyplot as plt
from horse1d import lab
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('Ready: 117 arteries, 62 tissue beds, one beating heart.')

## 1. Meet the arteries

Each line is one artery, drawn with the lengths and angles measured in the study. Thick lines are big arteries. The horse faces left: the head is up and to the left, the tail to the right, and the legs hang down. Left and right arteries sit on top of each other in this side view. The red dots are the five places we will measure.

In [ ]:
lab.plot_tree()
plt.show()

## 2. Run one horse

Pick the settings, then run the cell.

* **standing**: tick it for a standing horse, where gravity pulls on the blood. Untick it for a horse lying flat, where gravity does not matter.
* **heart_rate**: beats per minute. A horse at rest is near 40.
* **stiffness**: 1 is normal arteries. 1.5 means every artery is 1.5 times stiffer, as in an older horse.

The cell prints the key numbers, draws blood speed and blood pressure through one heartbeat at the five sites, and colors every artery by its average pressure.

In [ ]:
#@title Choose a horse, then run this cell
standing = False  #@param {type:"boolean"}
heart_rate = 40  #@param {type:"slider", min:35, max:80, step:1}
stiffness = 1.0  #@param {type:"slider", min:0.7, max:1.5, step:0.05}

horse = lab.run_horse(standing, heart_rate, stiffness)
if horse:
    lab.summary(horse)
    lab.plot_sites(horse)
    lab.plot_map(horse, "mean")
    plt.show()

## 3. Compare two horses

Set up horse A and horse B, for example the same horse lying down (A) and standing up (B). The two maps share one color scale, so you can compare colors directly. Choose what the maps show:

* **mean**: the average pressure over a heartbeat,
* **systolic**: the peak pressure of each beat,
* **pulse**: systolic minus diastolic, how much the pressure swings with each beat.

The last figure follows the pressure pulse down the aorta, from 26 cm to 86 cm from the heart.

In [ ]:
#@title Horse A and horse B
A_standing = False  #@param {type:"boolean"}
A_heart_rate = 40  #@param {type:"slider", min:35, max:80, step:1}
A_stiffness = 1.0  #@param {type:"slider", min:0.7, max:1.5, step:0.05}
B_standing = True  #@param {type:"boolean"}
B_heart_rate = 40  #@param {type:"slider", min:35, max:80, step:1}
B_stiffness = 1.0  #@param {type:"slider", min:0.7, max:1.5, step:0.05}
map_shows = "mean"  #@param ["mean", "systolic", "pulse"]

A = lab.run_horse(A_standing, A_heart_rate, A_stiffness)
B = lab.run_horse(B_standing, B_heart_rate, B_stiffness)
lab.summary(A, B)
lab.plot_sites(A, B)
lab.plot_maps(A, B, quantity=map_shows)
lab.plot_aorta(A, B)
plt.show()

## 4. Check gravity by hand

A column of blood *h* meters tall presses down with a pressure of ρ*gh*, where ρ = 1050 kg/m³ is the density of blood and *g* = 9.81 m/s². One mmHg is 133.3 Pa. The cell lists how high each site sits above the heart. Change `h` to the height difference you want to check.

In [ ]:
xy = lab.tree_coordinates()
for name, seg in lab.SITES.items():
    (x0, y0), (x1, y1) = xy[seg]
    print(f"{name:24s} {0.5 * (y0 + y1):+.2f} m above the heart")

rho, g = 1050, 9.81
h = 1.0   # meters: change this
print(f"\nrho * g * h = {rho * g * h:.0f} Pa = {rho * g * h / 133.3:.0f} mmHg")

## 5. The Womersley number

In a pulsing flow, the Womersley number α = *R* √(2π*f*ρ/μ) compares how fast the flow changes with each heartbeat to how fast friction can slow the blood near the wall. *R* is the artery's radius, *f* the heart rate in beats per second, ρ the blood's density and μ its viscosity (0.004 Pa·s). When α is large, the middle of the artery moves like a plug and friction acts only in a thin layer by the wall. The study needed the Womersley theory of friction because a horse's arteries are so big.

In [ ]:
import numpy as np
rho, mu = 1050, 0.004

def womersley(radius_m, beats_per_min):
    return radius_m * np.sqrt(2 * np.pi * (beats_per_min / 60) * rho / mu)

print("horse aorta, radius 3.4 cm, 40 beats/min:   ", round(womersley(0.034, 40), 1))
print("human aorta, radius 1.25 cm, 70 beats/min:  ", round(womersley(0.0125, 70), 1))
print("horse median artery, radius 3.5 mm, 40 bpm: ", round(womersley(0.0035, 40), 1))

## 6. Check the model against the study (optional)

The study reports these numbers for its own model of a resting horse (40 beats a minute), lying and standing. Ours are next to them. The study also measured some of them in real horses, listed at the bottom.

In [ ]:
lab.compare_with_paper()

## How the model works, in brief

* In every artery the model solves two equations along its length: one for **mass** (blood that flows in must flow out or stretch the wall) and one for **momentum** (pressure differences push the blood; friction at the wall and gravity hold it back).
* The artery walls are **elastic**, and they get stiffer as the pressure rises, following a formula the study took from measurements of human arteries.
* At the start of the aorta, a **beating left ventricle** squeezes and relaxes on a timed curve (a "time-varying elastance" heart).
* Where an artery branches, the **pressure is the same** in all branches and **no blood is lost**.
* At the 62 ends, each **tissue bed** is a resistor and a balloon (a 3-element Windkessel) with values from the study.

For the equations and every parameter, see *Horse_Artery_Model_Overview.pdf*. The model has no reflexes, so faster heart rates and stiffer arteries show what the mechanics alone would do.

## Credits

**Model and data:** Vera L, Campos Arias D, Muylle S, Stergiopulos N, Segers P, van Loon G (2019). A 1D computer model of the arterial circulation in horses: an important resource for studying global interactions between heart and vessels under normal and pathological conditions. *PLoS ONE* 14(8): e0221425. https://doi.org/10.1371/journal.pone.0221425 (CC BY 4.0).

**Python version and lesson:** EquiPhysics, a nonprofit in Sonoita, Arizona (equiphysics.org), 2026. Free under the MIT License: https://github.com/equiphysics/education/tree/main/labs/Horse-Artery-Lab

Did you use this lab with a class? Please tell us how it went: https://www.equiphysics.org/for-teachers#tell-us